# inference-mode-step — ex1: decorate step with inference_mode to allow in-place leaf update

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inference-mode-step`. Running the final beacon cell reports progress against the `PyTorch: Inference mode step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Inference mode step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inference-mode-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inference-mode-step"
DD_SUBTOPIC = "PyTorch: Inference mode step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `@t.inference_mode()` on `step` — quick refresher

PyTorch optimizers mutate parameters IN PLACE. The naive form `theta -= self.lr * g` on a leaf with `requires_grad=True` outside any no-grad context raises:

> *RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.*

The fix is to declare `step` as living outside autograd's bookkeeping. Two equivalent decorators:

```
@t.no_grad()           # disables grad tracking inside the call
@t.inference_mode()    # stricter newer version (also kills version counters)
```

ARENA's SGD/RMSprop/Adam impls all use `@t.inference_mode()` on `step`. That single decorator is what lets the body do `theta -= ...` and `buffer.copy_(...)` without autograd screaming. It is a hard requirement, not a stylistic choice.

### Exercise 1 — decorate step with inference_mode to allow in-place leaf update

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `@t.inference_mode()` decorator to a hand-rolled optimizer's `step` method so the in-place `theta -= lr * g` leaf-update succeeds without raising.
> Keywords: inference-mode, no-grad, in-place-leaf
> ```

**KCs targeted:** `inference-mode-decorator-wraps-step`, `inference-mode-allows-leaf-in-place-mutation`

Implement `Ex1InferenceSGD` — a minimal SGD optimizer whose `step` method is decorated with `@t.inference_mode()`.

1. `__init__(self, params, lr)`: materialize `params` into `self.params = list(params)`, store `self.lr = lr`.
2. `step(self)`: decorated with `@t.inference_mode()`. For each param `p` with non-None `.grad`, do the BARE in-place update `p -= self.lr * p.grad` — NOT `p.data -= ...`. The decorator is what allows this on a leaf with `requires_grad=True`.
3. `zero_grad(self)`: set every `p.grad = None`.

The test verifies:
- One step actually moves the weights toward the target.
- The decorator is present (a missing decorator would make the bare `p -= ...` raise the leaf-in-place error).
- Re-running the same forward AFTER `.step()` gives a smaller loss than before.

In [ ]:
class Ex1InferenceSGD:
    """SGD whose step is decorated with @t.inference_mode()."""

    def __init__(self, params, lr: float):
        raise NotImplementedError()

    def step(self):
        raise NotImplementedError()

    def zero_grad(self):
        raise NotImplementedError()


def _test_ex1():
    # Trivial regression: fit y = 3x with a single scalar weight.
    model = t.nn.Linear(1, 1, bias=False)
    with t.no_grad():
        model.weight.copy_(t.tensor([[0.0]]))
    opt = Ex1InferenceSGD(model.parameters(), lr=0.1)
    assert isinstance(opt.params, list), 'optimizer must materialize params into a list'

    x = t.tensor([[1.0], [2.0], [3.0], [4.0]])
    y = t.tensor([[3.0], [6.0], [9.0], [12.0]])

    loss_before = ((model(x) - y) ** 2).mean().item()
    loss = ((model(x) - y) ** 2).mean()
    loss.backward()
    opt.step()                      # must NOT raise the leaf-in-place error
    opt.zero_grad()
    loss_after = ((model(x) - y) ** 2).mean().item()

    assert loss_after < loss_before, (
        f'one SGD step should decrease loss: {loss_before:.4f} -> {loss_after:.4f}; '
        f'either the in-place update was silently a no-op or weights did not move'
    )
    # Param values actually moved.
    assert model.weight.item() != 0.0, (
        'weight should have moved off 0 after the step; '
        'is your step body actually mutating in place?'
    )

    # Verify the decorator was applied — the step function should
    # be wrapped (the bare in-place leaf update would otherwise raise).
    # We confirm this indirectly: run a SECOND step from a still-grad-required leaf.
    model2 = t.nn.Linear(1, 1, bias=False)
    with t.no_grad():
        model2.weight.copy_(t.tensor([[0.0]]))
    opt2 = Ex1InferenceSGD(model2.parameters(), lr=0.1)
    for _ in range(5):
        L = ((model2(x) - y) ** 2).mean()
        L.backward()
        opt2.step()                 # repeated bare in-place mutation
        opt2.zero_grad()
    assert abs(model2.weight.item() - 3.0) < 0.5, (
        f'after 5 steps weight should be approaching 3.0; got {model2.weight.item():.4f}; '
        f'is the decorator wrapping `step` correctly?'
    )
    # After all steps the param should still require grad (decorator did not contaminate it).
    assert model2.weight.requires_grad, (
        'param should still require grad after step; '
        'inference_mode should only affect the step call, not the param state'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class Ex1InferenceSGD:
    def __init__(self, params, lr: float):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None
```

**Why `p -= ...` and not `p.data -= ...`.** With `@t.inference_mode()` (or `@t.no_grad()`) wrapping `step`, the in-place op on a leaf is legal — autograd is told to ignore this block entirely. Without the decorator, the same line raises `a leaf Variable that requires grad is being used in an in-place operation.` The `.data` escape hatch is the older workaround you reach for OUTSIDE such a block.

**`@t.inference_mode()` vs `@t.no_grad()`.** Functionally interchangeable for the optimizer step. `inference_mode` is newer and slightly stricter — it disables version counters and forbids later upgrading the produced tensors to `requires_grad=True`. ARENA uses `inference_mode` because PyTorch recommends it for new code.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()